<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/template-report-Navier-Stokes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Navier-Stokes equations**
**Johan Hoffman**

# **Abstract**

This short report show an example on how to use FEniCSx to solve the Navier-Stokes equations, which is used in the course DD2365 Advanced Computation in Fluid Mechanics, at the KTH Royal Institute of Technology.

[DD2365 course website.](https://www.kth.se/social/course/DD2365/)

# **About the code**

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
# SPDX-License-Identifier: MIT
#
# FEniCSx port of the DD2365 Navier-Stokes notebook.
# Original FEniCS version: https://github.com/johanhoffman/DD2365
#
# This template is maintained by Johan Hoffman.
# Please report problems to jhoffman@kth.se

# **Set up environment**

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
import time
import ufl
import basix.ufl as bufl
from mpi4py import MPI
from matplotlib import pyplot as plt

import dolfinx
from dolfinx import fem
from dolfinx.fem import functionspace, Function, form, assemble_scalar, Constant
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc

In [ ]:
# --- canonical: gmsh_rect_minus_circles v2 ---
import math
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


def gmsh_rect_minus_circles(L, H, circles, resolution):
    """Rectangle [0,L]×[0,H] minus circular holes, meshed with gmsh OCC.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii to subtract.
    resolution : int
        Mesh density parameter.  The global size bound is

            lc = 0.65 * sqrt(L² + H²) / resolution

        This matches the mshr/CGAL semantics used in the legacy FEniCS
        notebooks: mshr's ``resolution`` sets a CGAL size bound equal to
        the bounding-box diagonal divided by ``resolution``; gmsh realised
        edges are approximately 0.6× that bound.  The factor 0.65 was
        calibrated so that the standard test case (L=4, H=2, 3 circular
        holes, resolution=32) produces ≈2319 cells — matching the legacy
        mshr mesh (2319 cells, 1247 P1 dofs).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Note
    ----
    Facet tags are not produced here.  Call ``tag_boundaries(msh, L, H)``
    on the final mesh (after any refinement) to obtain boundary MeshTags.
    """
    _ALPHA = 0.65
    lc = _ALPHA * math.sqrt(L**2 + H**2) / resolution

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    rect = gmsh.model.occ.addRectangle(0.0, 0.0, 0.0, L, H)
    disks = [(2, gmsh.model.occ.addDisk(cx, cy, 0.0, r, r)) for cx, cy, r in circles]
    if disks:
        gmsh.model.occ.cut([(2, rect)], disks)
    gmsh.model.occ.synchronize()

    gmsh.option.setNumber("Mesh.MeshSizeMax", lc)

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")

    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v2 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

# **Introduction**

The Navier-Stokes equations take the form

$\dot u + (u\cdot \nabla)u + \nabla p -\Delta u = f,\quad \nabla \cdot u=0,$

together with suitable initial and boundary conditions.

Here we present a FEniCSx implementation of a stabilized space-time finite element method to solve the Navier-Stokes equations in 2D. The solution is visualized using matplotlib, and is also exported as XDMF files which can be visualized in Paraview.

We seek a finite element approximation $(u,p)\in V\times Q$ such that

$(\dot u + (u\cdot \nabla)u, v) - (p,\nabla \cdot v) + (\nu \nabla u,\nabla v) + (\nabla \cdot u, q) + SD(u,p;v,q) = (f,v),$

for all test functions $(v,q) \in \hat V\times \hat Q$, where $SD(u,p;v,q)$ is a residual based stabilization term.

We present an example of flow past a circular cylinder, for which we compute the force on the surface of the cylinder $\Gamma$ in the direction $\phi$, by Green's formula:

$
F(u,p,\phi)= ~<\nu \nabla u\cdot n-pn, \Phi>_{\Gamma} ~=~(\dot u+(u\cdot \nabla)u, \Phi) + (\nu \nabla u,\nabla \Phi) - (p,\nabla \cdot\Phi)-(f,\Phi),
$

with $\Phi\in V$ a function for which $\Phi\vert _{\Gamma}=\phi$ and $\Phi\vert_{\partial \Omega \setminus \Gamma}=0$. With $\phi=(1,0)$ we get the drag force $F_D$, and with $\phi=(0,1)$ the lift force $F_L$. The drag and lift coefficients are obtained by normalization,

$
c_D = \frac{2F_D}{\rho U^2D}, \quad c_L = \frac{2F_L}{\rho U^2D}
$

where $\rho$ is the density (here $\rho=1$), $U$ the characteristic velocity (here $U=1$), and $D$ the characteristic length scale (here $D$ is the diameter of the cylinder).

The Reynolds number is defined as $Re=\frac{UD}{\nu}$

To read more about how to use similar methods for more complex problems, see e.g.

[Hoffman, Johan, et al. "Towards a parameter-free method for high reynolds number turbulent flow simulation based on adaptive finite element approximation." Computer Methods in Applied Mechanics and Engineering 288 (2015): 60-74.](https://www.sciencedirect.com/science/article/pii/S0045782514004836)

# **Method**

**Define domain and mesh**

In [ ]:
# Define rectangular domain
L = 4.0
H = 2.0

# Define circle
cx = 1.0
cy = 0.5 * H
cr = 0.2

# Define subdomains (for boundary conditions)
no_levels = 0
resolution = 32
uin = 1.0

msh, cell_tags = gmsh_rect_minus_circles(L, H, [(cx, cy, cr)], resolution)

for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda mp: np.sqrt((mp[:, 0] - cx)**2 + (mp[:, 1] - cy)**2) < 1.0
    )

facet_tags = tag_boundaries(msh, L, H)

tdim = msh.topology.dim
fdim = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local
print(f"Mesh: {n_cells} cells")

plot_mesh(msh, title=f"Mesh: {n_cells} cells")

**Define finite element approximation spaces**

In [ ]:
# Generate finite element spaces (for velocity and pressure)
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
V = functionspace(msh, P1v)
Q = functionspace(msh, P1s)

# Define trial and test functions
u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

h_arr = msh.h(tdim, np.arange(n_cells, dtype=np.int32))
hmin = float(h_arr.min())
print(f"V dofs: {V.dofmap.index_map.size_global * V.dofmap.index_map_bs}, "
      f"Q dofs: {Q.dofmap.index_map.size_global}, hmin = {hmin:.6f}")

**Define boundary conditions**

In [ ]:
# Define boundary conditions
V0, _ = V.sub(0).collapse()
V1, _ = V.sub(1).collapse()

left_facets  = facet_tags.find(1)
right_facets = facet_tags.find(2)
lower_facets = facet_tags.find(3)
upper_facets = facet_tags.find(4)
obj_facets   = facet_tags.find(5)

def _bc_vel(val, sub_idx, sub_V, facets):
    f = Function(sub_V); f.x.array[:] = val
    dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
    return fem.dirichletbc(f, dofs, V.sub(sub_idx))

bcu_in0  = _bc_vel(uin, 0, V0, left_facets)
bcu_in1  = _bc_vel(0.0, 1, V1, left_facets)
bcu_upp1 = _bc_vel(0.0, 1, V1, upper_facets)
bcu_low1 = _bc_vel(0.0, 1, V1, lower_facets)
bcu_obj0 = _bc_vel(0.0, 0, V0, obj_facets)
bcu_obj1 = _bc_vel(0.0, 1, V1, obj_facets)
bcu = [bcu_in0, bcu_in1, bcu_upp1, bcu_low1, bcu_obj0, bcu_obj1]

bcp_out = fem.dirichletbc(
    Constant(msh, PETSc.ScalarType(0.0)),
    fem.locate_dofs_topological(Q, fdim, right_facets), Q)
bcp = [bcp_out]

# **Results**

**Define flow parameters**

In [ ]:
# Set viscosity
nu = 4.0e-3

**Define method parameters**

In [ ]:
# Define iteration functions
# (u0,p0) solution from previous time step
# (u1,p1) linearized solution at present time step
u0 = Function(V, name="u0")
u1 = Function(V, name="u1")
p1 = Function(Q, name="p1")

# Set parameters for nonlinear and linear solvers
num_nnlin_iter = 5

# Time step length
dt = 0.5 * hmin

# Time stepping
T = 30.0
plot_freq = 10

**Define variational problem**

In [ ]:
# Define variational problem

# Stabilization parameters
h_cell = ufl.CellDiameter(msh)
u_mag = ufl.sqrt(ufl.dot(u1, u1))
d1 = 1.0 / ufl.sqrt((1.0/dt)**2 + (u_mag/h_cell)**2)
d2 = h_cell * u_mag

# Mean velocities for trapezoidal time stepping
um  = 0.5 * (u + u0)   # contains TrialFunction
um1 = 0.5 * (u1 + u0)  # known linearization point

# Momentum variational equation on residual form
Fu = (ufl.inner((u - u0)/dt + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
    - p1*ufl.div(v)*ufl.dx
    + nu*ufl.inner(ufl.grad(um), ufl.grad(v))*ufl.dx
    + d1*ufl.inner((u - u0)/dt + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                   ufl.dot(ufl.grad(v), um1))*ufl.dx
    + d2*ufl.div(um)*ufl.div(v)*ufl.dx)
au = ufl.lhs(Fu)
Lu = ufl.rhs(Fu)

# Continuity variational equation on residual form
Fp = (d1*ufl.inner((u1 - u0)/dt + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                   ufl.grad(q))*ufl.dx
    + ufl.div(um1)*q*ufl.dx)
ap = ufl.lhs(Fp)
Lp = ufl.rhs(Fp)

a_u = form(au); l_u = form(Lu)
a_p = form(ap); l_p = form(Lp)
print("Forms compiled.")

**Triple decomposition**

In [ ]:
import scipy.linalg.lapack as la


def triple_decomposition(grad_u):
    # dtype=float: legacy used integer literal zeros giving int dtype, which truncated
    # the gradient values to integers — that is a bug; the fix is np.zeros((3,3)).
    new_grad = np.zeros((3, 3))
    for i in range(2):
        for j in range(2):
            new_grad[i, j] = grad_u[i, j]
    def dselect(arg1, arg2): return (arg2 == 0)
    T = la.dgees(dselect, new_grad, sort_t=1)[0]
    sh = np.linalg.norm([T[0, 1], T[0, 2], T[1, 2] + T[2, 1]])
    el = np.linalg.norm(np.diag(T))
    rr = np.sqrt(2 * min(abs(T[1, 2]), abs(T[2, 1]))**2)
    return sh, el, rr


# L2 projection setup for nabla_grad(u1) — mass matrix assembled once,
# RHS reassembled each plot step (u1 changes each step).
_V2 = functionspace(msh, ("Lagrange", 1))
_u_s = ufl.TrialFunction(_V2); _v_s = ufl.TestFunction(_V2)
_a_proj = form(_u_s * _v_s * ufl.dx)
_A_proj = create_matrix(_a_proj); assemble_matrix(_A_proj, _a_proj); _A_proj.assemble()
_ksp_proj = PETSc.KSP().create(msh.comm)
_ksp_proj.setOperators(_A_proj); _ksp_proj.setType("cg")
_ksp_proj.getPC().setType("icc")
_ksp_proj.setTolerances(rtol=1e-10); _ksp_proj.setFromOptions()

_G00 = Function(_V2); _G01 = Function(_V2)
_G10 = Function(_V2); _G11 = Function(_V2)
_shear      = Function(_V2, name="shear")
_elongation = Function(_V2, name="elongation")
_rotation   = Function(_V2, name="rotation")
_divu       = Function(_V2, name="divu")


def _project_comp(expr, G):
    b = create_vector(_V2)
    with b.localForm() as loc: loc.set(0.0)
    assemble_vector(b, form(expr * _v_s * ufl.dx))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    _ksp_proj.solve(b, G.x.petsc_vec)
    G.x.scatter_forward()


def _do_triple_decomposition():
    nG = ufl.nabla_grad(u1)
    _project_comp(nG[0, 0], _G00); _project_comp(nG[0, 1], _G01)
    _project_comp(nG[1, 0], _G10); _project_comp(nG[1, 1], _G11)
    g00 = _G00.x.array.real; g01 = _G01.x.array.real
    g10 = _G10.x.array.real; g11 = _G11.x.array.real
    n_v = _V2.dofmap.index_map.size_local
    for v in range(n_v):
        gm = np.array([[g00[v], g01[v]], [g10[v], g11[v]]])
        sh, el, rr = triple_decomposition(gm)
        _shear.x.array[v] = sh; _elongation.x.array[v] = el; _rotation.x.array[v] = rr
    _divu.x.array[:n_v] = g00[:n_v] + g11[:n_v]

**Compute force on boundary**

Note: the force is sampled once per time step after the nonlinear loop converges; the legacy implementation sampled at every nonlinear iteration.

In [ ]:
# Define the direction of the force to be computed
phi_x = 0.0
phi_y = 1.0

psi = Function(V, name="psi")
psi.x.array[:] = 0.0

# Set phi on cylinder (tag 5) dofs using locate_dofs_topological
dofs_x = fem.locate_dofs_topological((V.sub(0), V0), fdim, obj_facets)
dofs_y = fem.locate_dofs_topological((V.sub(1), V1), fdim, obj_facets)
if len(dofs_x):
    psi.x.array[dofs_x[0]] = phi_x
if len(dofs_y):
    psi.x.array[dofs_y[0]] = phi_y
psi.x.scatter_forward()

# Force by Green's formula (volume form)
Force = (ufl.inner((u1 - u0)/dt + ufl.dot(ufl.grad(um1), um1), psi)*ufl.dx
         - p1*ufl.div(psi)*ufl.dx
         + nu*ufl.inner(ufl.grad(um1), ufl.grad(psi))*ufl.dx)
force_form = form(Force)

# Force normalization
D = 2 * cr
normalization = -2.0 / D

**Set plotting variables and open export files**

In [ ]:
# Open XDMF export
xdmf = XDMFSeries("results-nse/solution.xdmf")

# Preallocate matrices and vectors
A_u = create_matrix(a_u); b_u = create_vector(V)
A_p = create_matrix(a_p); b_p = create_vector(Q)


def _make_ksp_ilu(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("bcgs"); ksp.getPC().setType("ilu")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
    return ksp


def _make_ksp_amg(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("bcgs")
    pc = ksp.getPC(); pc.setType("hypre"); pc.setHYPREType("boomeramg")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
    return ksp


ksp_u = _make_ksp_ilu(A_u)
ksp_p = _make_ksp_amg(A_p)

# Set plot frequency
plot_time = 0.0

# Force computation data
force_array = []
time_arr = []
start_sample_time = 1.0

**Time stepping algorithm**

In [ ]:
# Initialise boundary conditions on u0 and u1
set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

# Time stepping
t_wall0 = time.time()
t = dt
step = 0
while t < T + 1e-10:

    # Solve non-linear problem
    k = 0
    while k < num_nnlin_iter:

        # Assemble momentum matrix and vector
        A_u.zeroEntries()
        assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
        with b_u.localForm() as loc: loc.set(0.0)
        assemble_vector(b_u, l_u)
        apply_lifting(b_u, [a_u], bcs=[bcu])
        b_u.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_u, bcu)
        ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
        u1.x.scatter_forward(); set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

        # Assemble continuity matrix and vector
        A_p.zeroEntries()
        assemble_matrix(A_p, a_p, bcs=bcp); A_p.assemble()
        with b_p.localForm() as loc: loc.set(0.0)
        assemble_vector(b_p, l_p)
        apply_lifting(b_p, [a_p], bcs=[bcp])
        b_p.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_p, bcp)
        ksp_p.setOperators(A_p, A_p); ksp_p.solve(b_p, p1.x.petsc_vec)
        p1.x.scatter_forward(); set_bc(p1.x.petsc_vec, bcp); p1.x.scatter_forward()

        k += 1

    # Compute force (sampled once per step after nonlinear convergence)
    F_val = float(msh.comm.allreduce(assemble_scalar(force_form), op=MPI.SUM))
    if t > start_sample_time:
        force_array.append(normalization * F_val)
        time_arr.append(t)

    if t > plot_time:

        s = "Time t = " + repr(t)
        print(s)

        xdmf.write([u1, p1], t)

        # Triple decomposition
        _do_triple_decomposition()

        # Plot Triple Decomposition
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for _ax, _func, _title in zip(axes,
                [_shear, _elongation, _rotation],
                ["Shear", "Strain", "Rotation"]):
            plot_scalar(_func, title=_title, ax=_ax)
        plt.tight_layout(); plt.show()

        # Plot divergence
        fig2, ax2 = plt.subplots(figsize=(8, 3))
        plot_scalar(_divu, title="divergence", ax=ax2)
        plt.tight_layout(); plt.show()

        # Plot solution
        plot_vector(u1, title="Velocity")
        plot_scalar(p1, title="Pressure")

        plt.figure()
        plt.title("Force")
        if time_arr:
            plt.plot(time_arr, force_array)
        plt.show()

        plot_time += T / plot_freq

    # Update time step
    u0.x.array[:] = u1.x.array
    t += dt
    step += 1

xdmf.close()
t_wall = time.time() - t_wall0
print(f"Done: {step} steps, wall time = {t_wall:.1f}s")

In [ ]:
u_L2 = float(np.sqrt(assemble_scalar(form(ufl.inner(u1, u1) * ufl.dx))))
p_L2 = float(np.sqrt(assemble_scalar(form(p1**2 * ufl.dx))))
print(f"||u1||_L2 = {u_L2:.6f}")
print(f"||p1||_L2 = {p_L2:.6f}")
print(f"cells     = {n_cells}")
print(f"dt        = {dt:.6f}")
print(f"steps     = {step}")

# **Discussion**

A stabilized finite element method was implemented in FEniCSx to solve the Navier-Stokes equations in 2D. The method was tested for the model problem of flow past a circular obstacle, and for a high enough Reynolds number and sufficient mesh resolution [a von Karman vortex street developed as expected.](https://en.wikipedia.org/wiki/K%C3%A1rm%C3%A1n_vortex_street)